# RAG项目案例
以某东商品衣服为例，以衣服属性构建本地知识。可以自由更新本地知识，用户问题的答案也是基于本地知识生成的。
## 00. 这份Notebook是干什么的

将几份文本资料放进“知识库”，再让大模型回答问题时先查资料、再回答。

用处有 3 个：
1. 从 0 到 1 跑通这个 RAG 案例
2. 知道每个 Python 文件分别负责什么
3. 以后能把自己的文本资料替换进去，做成自己的知识库

## 01. 先看项目里有什么

这一步只是认目录。先知道项目里有哪些 `.py` 文件、有哪些示例 `.txt` 文件，后面就不会迷路。

In [ ]:
import os
import sys
from pathlib import Path


def locate_project_root() -> Path:
    signatures = [("app_qa.py", "knowledge_base.py", "data/尺码推荐.txt")]
    seen: set[Path] = set()
    candidates: list[Path] = []

    def add_candidate(path: Path) -> None:
        resolved = path.resolve()
        if resolved not in seen and resolved.exists() and resolved.is_dir():
            seen.add(resolved)
            candidates.append(resolved)

    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        add_candidate(base)
        try:
            child_dirs = [child for child in base.iterdir() if child.is_dir()]
        except PermissionError:
            child_dirs = []
        for child in child_dirs:
            add_candidate(child)
            try:
                grandchild_dirs = [grandchild for grandchild in child.iterdir() if grandchild.is_dir()]
            except PermissionError:
                grandchild_dirs = []
            for grandchild in grandchild_dirs:
                add_candidate(grandchild)

    for candidate in candidates:
        if all((candidate / rel).exists() for rel in signatures[0]):
            return candidate

    raise FileNotFoundError("没有找到 03_RAG项目案例 目录，请确认从仓库目录中打开 Notebook。")


project_root = locate_project_root()
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

(project_root / 'chat_history').mkdir(exist_ok=True)
(project_root / 'chroma_db').mkdir(exist_ok=True)
print(f'当前工作目录: {project_root}')

python_files = sorted(p.name for p in project_root.glob('*.py'))
data_files = sorted(p.name for p in (project_root / 'data').glob('*.txt'))

print('项目中的 Python 文件:')
for name in python_files:
    print('-', name)

print('项目中的示例知识库文件:')
for name in data_files:
    print('-', name)


## 02. 安装运行依赖

如果是第一次跑这个案例，先执行这一格。它会安装 Streamlit、LangChain、Chroma、DashScope 等依赖。

如果之前已经装过，也可以跳过。

In [ ]:
%pip install streamlit langchain-core langchain-community langchain-chroma langchain-text-splitters chromadb dashscope

## 03. 检查 API Key

这个项目要调用阿里云百炼，所以必须先有 `DASHSCOPE_API_KEY`。

没有它，就像手机没插 SIM 卡，后面的联网能力都不能用。

In [ ]:
import os
import sys

# 看看 Python 版本
print('Python 版本:', sys.version.split()[0])

# 看看环境变量里有没有 API Key
api_key_exists = bool(os.getenv('DASHSCOPE_API_KEY'))
print('DASHSCOPE_API_KEY 是否已设置:', api_key_exists)

# 没有 Key 就先停下，因为后面肯定跑不通
if not api_key_exists:
    raise EnvironmentError('请先设置 DASHSCOPE_API_KEY，再继续运行后面的单元。')

## 04. 导入项目里的现成模块

这一节很重要。这里不是重写代码，而是直接复用项目里已经写好的模块。

可以先这样理解它们：
- `knowledge_base.py`：负责把文本放进知识库
- `vector_stores.py`：负责从知识库里查资料
- `rag.py`：负责把“查资料 + 问大模型”串起来
- `config_data.py`：负责放配置

In [ ]:
import json
import shutil

# 导入项目中的配置和服务类
import config_data as config
from knowledge_base import KnowledgeBaseService
from vector_stores import VectorStoreService
from rag import RagService
from langchain_community.embeddings import DashScopeEmbeddings

# 打印关键配置，先有个整体印象
print('embedding 模型:', config.embedding_model_name)
print('chat 模型:', config.chat_model_name)
print('向量库目录:', config.persist_directory)
print('集合名:', config.collection_name)
print('检索返回条数 k:', config.similarity_threshold)

## 05. 可选：清空旧数据

如果想从头复现，最好先把旧的向量库、聊天记录、去重记录清空。

为了安全，这里默认不删除。只有你把 `RESET_DEMO_STATE` 改成 `True` 才会执行。

In [ ]:
RESET_DEMO_STATE = False#True

# 这 3 个位置分别存放向量库、聊天记录、去重记录
targets = [Path(config.persist_directory), Path('chat_history'), Path(config.md5_path)]

if RESET_DEMO_STATE:
    for target in targets:
        if target.is_dir():
            shutil.rmtree(target)
            print(f'[已删除目录] {target}')
        elif target.exists():
            target.unlink()
            print(f'[已删除文件] {target}')
        else:
            print(f'[跳过] {target} 不存在')
else:
    print('当前是安全模式，没有删除任何文件。')
    print('如果你要从头复现，把 RESET_DEMO_STATE 改成 True 后再运行一次。')

## 06. 看看示例文本里写了什么

在导入知识库之前，先看看原始文本内容。这样就会知道后面模型到底是根据什么资料来回答问题的。

In [ ]:
data_dir = Path('data')
txt_files = sorted(data_dir.glob('*.txt'))

# 只预览每个文件的前 300 个字符
for txt_file in txt_files:
    print(f'===== {txt_file.name} =====')
    print(txt_file.read_text(encoding='utf-8')[:300])
    print('-' * 40)

## 07. 把文本放进知识库

这一步就是把 `data` 目录下的文本喂给程序。程序会自动做 4 件事：
1. 读取文本
2. 必要时切分文本
3. 做向量化
4. 写入 Chroma 向量库

In [ ]:
# 创建知识库服务对象
kb_service = KnowledgeBaseService()
import_results = []
# 逐个导入 txt 文件
for txt_file in txt_files:
    text = txt_file.read_text(encoding='utf-8')
    result = kb_service.upload_by_str(text, txt_file.name)
    import_results.append((txt_file.name, result))
    print(f'{txt_file.name}: {result}')

print('一共处理了多少个文件:', len(import_results))

## 08. 看看本地有没有真的生成东西

刚开始在这里会疑惑：“我刚刚导入的东西到底存哪了？”

答案是：向量数据会写进 `chroma_db`，去重信息会写进 `md5.text`。下面直接看。

In [ ]:
from pathlib import Path
import config_data as config

persist_path = Path(config.persist_directory)
md5_path = Path(config.md5_path)

print('chroma_db 是否存在:', persist_path.exists())
print('md5.txt 是否存在:', md5_path.exists())

if persist_path.exists():
    print('chroma_db 下的文件:')
    for item in sorted(persist_path.rglob('*')):
        if item.is_file():
            print('-', item.resolve())

if md5_path.exists():
    print('md5 记录条数:', len(md5_path.read_text(encoding='utf-8').splitlines()))

## 09. 不问大模型，先单独测试“查资料”

RAG 分两步：先查资料，再回答。

这一节先只测“查资料”这一步，确认知识库检索是正常的。

In [ ]:
# 创建 embedding 和 retriever
embedding = DashScopeEmbeddings(model=config.embedding_model_name)
retriever = VectorStoreService(embedding=embedding).get_retriever()

query = '我身高178cm，体重160斤，应该推荐什么尺码？'
docs = retriever.invoke(query)

print('问题:', query)
print('检索到的文档数量:', len(docs))

for idx, doc in enumerate(docs, start=1):
    print(f'--- 文档 {idx} ---')
    print(doc.page_content[:300])
    print('metadata =', doc.metadata)

## 10. 跑一次完整的 RAG 问答

现在才是真正的“先查资料，再回答”。

这里的 `RagService` 会自动把检索、拼 Prompt、调用大模型这几步连起来。

In [ ]:
# 创建 RAG 服务对象
rag_service = RagService()
session_config = {'configurable': {'session_id': 'notebook_demo_user'}}

question = '针织毛衣如何保养？'
answer = rag_service.chain.invoke({'input': question}, session_config)

print('问题:', question)
print('回答:', answer)

## 11. 上下文记忆

这个案例不是只能一问一答，还会把聊天记录保存在本地，所以同一个 `session_id` 下，多轮提问是能接上文的。

In [ ]:
# 用同一个 session_id 连续问 3 次
multi_turn_questions = [
    '我身高178cm，体重160斤，适合什么尺码？',
    '如果我是黄皮，要去面试，颜色上怎么选？',
    '那针织类衣服平时怎么洗比较稳妥？'
]

for idx, question in enumerate(multi_turn_questions, start=1):
    answer = rag_service.chain.invoke({'input': question}, session_config)
    print(f'第 {idx} 轮问题: {question}')
    print('回答:', answer)
    print('-' * 60)

## 12. 聊天记录保存

如果想知道“为什么能记住我刚才问过什么”，就看这里。

聊天历史会被写进 `chat_history` 目录。

In [ ]:
history_file = Path('chat_history') / 'notebook_demo_user'
print('聊天历史文件路径:', history_file)

if history_file.exists():
    history_data = json.loads(history_file.read_text(encoding='utf-8'))
    print('消息条数:', len(history_data))
    print('前两条消息示例:')
    print(json.dumps(history_data[:2], ensure_ascii=False, indent=2))
else:
    print('还没有历史文件。请先运行上一节的多轮对话。')

## 13. 每个 Python 文件分别是干什么的

这是最适合小白建立全局认识的一节。你不用一开始就读源码，先记住“谁负责什么”就够了。

In [ ]:
module_map = {
    'config_data.py': '放模型名、路径、切分参数等配置',
    'knowledge_base.py': '把文本切分后写入向量库',
    'vector_stores.py': '把向量库包装成检索器 retriever',
    'rag.py': '把检索 + Prompt + 大模型串起来',
    'file_history_store.py': '把聊天记录保存到本地文件',
    'app_file_uploader.py': '提供上传 txt 文件的网页入口',
    'app_qa.py': '提供智能问答网页入口',
}

for name, desc in module_map.items():
    print(f'{name}: {desc}')

## 14. 如果不用 Notebook，怎么直接跑网页

这个项目有两个 Streamlit 页面。

一个页面负责上传资料，一个页面负责提问聊天。可以在终端里直接运行它们：

先切换到当前路径（路径可以直接去文件夹或vscode中复制，也可以在最开始的单元格查看）

我的路径是

cd d:\Agent\03-RAG\class\代码\AI大模型RAG与智能体开发\P4_RAG项目案例

powershell

streamlit run .\app_file_uploader.py

每次运行切都要切到代码存在的目录

powershell

streamlit run .\app_qa.py